In [ ]:
import pandas as pd

In [ ]:
feat_counts_path="/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/RF_models_automation/RF_results/raw/cluster_phyloglm_model_Feature_importance.tsv"
df = pd.read_csv(feat_counts_path, sep='\t', usecols=[0,2],names=['feature','gini_imp'],skiprows=1).reset_index(names='rank')
df

In [ ]:
df_phylo_res = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/Microviridae_analysis/multiple_cluster_analysis/combined_model_phyloglm/phyloglm_results_combined_model_pruned_full.csv",index_col=0)
df_phylo_res

In [ ]:
df_phylo_res.sort_values('z.value',ascending=False,inplace=True)
df_phylo_res[['pos','AA']] = df_phylo_res['feature'].str.split('_',expand=True)
df_phylo_res = df_phylo_res.loc[df_phylo_res['padj'] <= 0.001]
df_phylo_res.reset_index(names='rank',inplace=True)
df_phylo_res

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d

plt.rcParams.update({'font.size': 14})   # 🔥 global fontsize

def plot_cumulative_AA_by_sign_annotated(
    df,
    aa_groups=None,
    max_rank=100,
    figsize=(14,8),
    alpha=0.7,
    annotate_offset=(10, 0),   # (x_pixels, y_pixels) for text offset
    return_fig=True
):
    """
    Plot cumulative counts by feature rank for each amino acid, with two lines per AA:
      - positive (Estimate > 0): cumulative count above zero (solid)
      - negative (Estimate < 0): cumulative count below zero (dashed, mirrored)
    Annotates each line with the amino acid letter near the line end.
    Only ranks <= max_rank are considered.
    Returns fig, ax if return_fig True, else returns ax.
    """
    # Default groups and colors
    if aa_groups is None:
        aa_groups = [
            ("Positive charged", ["R", "H", "K"], "crimson"),
            ("Negative charged", ["D", "E"], "royalblue"),
            ("Polar uncharged", ["S", "T", "N", "Q"], "seagreen"),
            ("Hydrophobic", ["A", "V", "I", "L", "M"], "goldenrod"),
            ("Aromatic", ["F", "Y", "W"], "purple"),
            ("Special cases", ["C", "U", "G", "P"], "darkorange")
        ]

    # AA -> color map
    aa_to_color = {aa: color for _, aas, color in aa_groups for aa in aas}

    # Clean: drop NaNs in essential columns, ensure numeric rank
    df_clean = (
        df.dropna(subset=["rank", "AA", "Estimate"])
          .copy()
    )
    df_clean['rank'] = pd.to_numeric(df_clean['rank'], errors='coerce')
    df_clean = df_clean.dropna(subset=['rank'])
    # Filter to ranks <= max_rank
    df_clean = df_clean[df_clean['rank'] <= max_rank].sort_values('rank').reset_index(drop=True)

    # If no data after filtering, return quietly
    if df_clean.shape[0] == 0:
        raise ValueError(f"No data with rank <= {max_rank} after filtering NaNs.")

    # sign
    df_clean['sign'] = np.where(df_clean['Estimate'] < 0, 'negative', 'positive')

    # Build cumulative records for every AA and sign
    cum_list = []
    unique_ranks = df_clean['rank'].values  # keep same x for all
    for aa in np.unique(df_clean['AA'].astype(str)):
        for sign in ['positive', 'negative']:
            mask = ((df_clean['AA'] == aa) & (df_clean['sign'] == sign))
            cum = mask.cumsum().astype(int)
            if sign == 'negative':
                cum = -cum
            tmp = pd.DataFrame({
                'rank': unique_ranks,
                'AA': aa,
                'sign': sign,
                'cum_count': cum
            })
            cum_list.append(tmp)
    cum_df = pd.concat(cum_list, ignore_index=True)

    # Keep only series that actually have a nonzero value (check abs max)
    counts_by_series = cum_df.groupby(['AA','sign'])['cum_count'].agg(lambda x: x.abs().max())
    valid = counts_by_series[counts_by_series > 0].reset_index()[['AA','sign']]

    fig, ax = plt.subplots(figsize=figsize)

    # Plot per group to preserve color mapping and order
    for grp_name, aas, color in aa_groups:
        for aa in aas:
            for sign in ['positive','negative']:
                if not ((valid['AA'] == aa) & (valid['sign'] == sign)).any():
                    continue
                sub = cum_df[(cum_df['AA'] == aa) & (cum_df['sign'] == sign)]
                window = 20  # or 100 for stronger smoothing
                # sub['cum_count_smooth'] = sub['cum_count'].rolling(window, center=True, min_periods=1).mean()     # rolling average
                sub['cum_count_smooth'] = savgol_filter(sub['cum_count'], window_length=20, polyorder=3)           # Savitzky–Golay filter : Fits local polynomials → smooths without flattening peaks.

                linestyle = '-' if sign == 'positive' else '--'
                ax.plot(sub['rank'], sub['cum_count_smooth'],
                        linestyle=linestyle,
                        color=color,
                        linewidth=1.6,
                        alpha=alpha,
                        zorder=2)

                # annotate near the last non-NaN point (within the plotted range)
                # find the last index where cum_count != 0 (or use last point)
                nonzero = sub['cum_count'].to_numpy()
                if np.any(nonzero != 0):
                    last_idx = np.where(nonzero != 0)[0][-1]
                else:
                    last_idx = len(sub)-1

                # Get x, y for annotation
                x_val = sub['rank'].iloc[last_idx]
                y_val = sub['cum_count'].iloc[last_idx]

                # offset annotation in data coordinates by converting pixel offset
                # simpler approach: small relative offset based on axis scale
                x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
                y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
                dx = (annotate_offset[0] / fig.dpi) * (x_range / fig.get_size_inches()[0])  # approx
                dy = (annotate_offset[1] / fig.dpi) * (y_range / fig.get_size_inches()[1])  # approx

                ha = 'left'
                va = 'center'
                # # If near right edge, shift left
                # if x_val > (ax.get_xlim()[1] - 0.05 * x_range):
                #     ha = 'right'
                #     dx = -abs(dx)

                ax.text(x_val + dx, y_val + dy, aa,
                        color=color, fontsize=9, fontweight='bold',
                        ha=ha, va=va, alpha=0.9,
                        bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.0),
                        zorder=3)

    ax.axhline(0, color='black', linewidth=1)
    ax.set_xlim(left=df_clean['rank'].min(), right=max_rank)
    ax.set_xlabel('Feature rank')
    ax.set_ylabel('Cumulative count (positive above, negative below)')
    ax.set_title(f'Cumulative AA counts by rank (annotated), ranks ≤ {max_rank}')
    ax.grid(axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()

    return plt


In [ ]:
plt = plot_cumulative_AA_by_sign_annotated(df_phylo_res)
plt.savefig("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/RF_models_automation/RF_results/Plots/cluster_phyloglm_model/cum_counts_AA_100_feat.png", dpi=300)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

plt.rcParams.update({'font.size': 14})

def plot_cumulative_AA_by_sign_top_initial_slope(
    df,
    aa_groups=None,
    max_rank=500,
    figsize=(14,8),
    alpha=0.7,
    annotate_offset=(10, 0),
    return_fig=True,
    smoothing_window=21,       # Savitzky-Golay window (will be forced odd and <= series length)
    savgol_polyorder=3,
    top_n_per_direction=5,    # choose top N positive and top N negative
    n_init=50                 # compute slope on the first n_init ranks (use available length if shorter)
):
    """
    Plot cumulative AA counts with two lines per AA (pos/neg). Highlight the top N series
    selected by the slope computed over the **first n_init ranks** of the (smoothed) series.
    """
    if aa_groups is None:
        aa_groups = [
            ("Positive charged", ["R", "H", "K"], "crimson"),
            ("Negative charged", ["D", "E"], "royalblue"),
            ("Polar uncharged", ["S", "T", "N", "Q"], "seagreen"),
            ("Hydrophobic", ["A", "V", "I", "L", "M"], "goldenrod"),
            ("Aromatic", ["F", "Y", "W"], "purple"),
            ("Special cases", ["C", "U", "G", "P"], "darkorange")
        ]

    aa_to_color = {aa: color for _, aas, color in aa_groups for aa in aas}
    grey_color = "lightgray"

    # === Clean input ===
    df_clean = df.dropna(subset=["rank", "AA", "Estimate"]).copy()
    df_clean['rank'] = pd.to_numeric(df_clean['rank'], errors='coerce')
    df_clean = df_clean.dropna(subset=['rank'])
    df_clean = df_clean[df_clean['rank'] <= max_rank].sort_values('rank').reset_index(drop=True)
    if df_clean.shape[0] == 0:
        raise ValueError(f"No data with rank <= {max_rank} after filtering NaNs.")
    df_clean['sign'] = np.where(df_clean['Estimate'] < 0, 'negative', 'positive')

    # === Build cumulative series ===
    ranks = df_clean['rank'].values
    cum_rows = []
    for aa in np.unique(df_clean['AA'].astype(str)):
        for sign in ['positive', 'negative']:
            mask = ((df_clean['AA'] == aa) & (df_clean['sign'] == sign)).astype(int)
            cum = np.cumsum(mask).astype(int)
            if sign == 'negative':
                cum = -cum
            cum_rows.append(pd.DataFrame({
                'rank': ranks,
                'AA': aa,
                'sign': sign,
                'cum_count': cum
            }))
    cum_df = pd.concat(cum_rows, ignore_index=True)

    # filter out series that are all zero
    counts_by_series = cum_df.groupby(['AA','sign'])['cum_count'].agg(lambda x: x.abs().max())
    valid = counts_by_series[counts_by_series > 0].reset_index()[['AA','sign']]

    # === For each valid series compute smoothed series and slope on first n_init points ===
    series_list = []
    for _, row in valid.iterrows():
        aa = row['AA']
        sign = row['sign']
        sub = cum_df[(cum_df['AA'] == aa) & (cum_df['sign'] == sign)].copy().reset_index(drop=True)
        y = sub['cum_count'].to_numpy(dtype=float)

        # ensure appropriate window (odd, <= len)
        w = int(smoothing_window)
        if w < 3:
            w = 3
        if w % 2 == 0:
            w += 1
        if w > len(y):
            w = len(y) if (len(y) % 2 == 1) else max(1, len(y)-1)
        if w >= 3 and len(y) >= 3:
            try:
                y_smooth = savgol_filter(y, window_length=w, polyorder=savgol_polyorder)
            except Exception:
                y_smooth = pd.Series(y).rolling(window=max(1, w), center=True, min_periods=1).mean().to_numpy()
        else:
            y_smooth = y.copy()

        # compute slope on the first n_init points (or fewer if series shorter)
        m_len = min(n_init, len(y_smooth))
        if m_len >= 2 and not np.allclose(y_smooth[:m_len], y_smooth[:m_len][0]):
            x_init = sub['rank'].to_numpy(dtype=float)[:m_len]
            y_init = y_smooth[:m_len]
            # linear fit y = m*x + b
            m, b = np.polyfit(x_init, y_init, 1)
            slope_init = m
        else:
            slope_init = 0.0

        series_list.append({
            'AA': aa,
            'sign': sign,
            'rank': sub['rank'].to_numpy(),
            'y_raw': y,
            'y_smooth': y_smooth,
            'slope_init': slope_init
        })

    # === select top series by initial slope ===
    slopes_df = pd.DataFrame([{'AA': s['AA'], 'sign': s['sign'], 'slope_init': s['slope_init']} for s in series_list])

    pos_candidates = slopes_df[slopes_df['slope_init'] > 0].sort_values('slope_init', ascending=False).head(top_n_per_direction)
    neg_candidates = slopes_df[slopes_df['slope_init'] < 0].sort_values('slope_init', ascending=True).head(top_n_per_direction)

    highlight_set = set(tuple(x) for x in pd.concat([pos_candidates, neg_candidates])[['AA','sign']].to_numpy())

    # === Plot ===
    fig, ax = plt.subplots(figsize=figsize)

    for s in series_list:
        aa, sign = s['AA'], s['sign']
        x = s['rank']
        y = s['y_smooth']
        is_highlight = (aa, sign) in highlight_set
        color = aa_to_color.get(aa, 'black') if is_highlight else grey_color
        linestyle = '-' if sign == 'positive' else '--'
        linewidth = 2.2 if is_highlight else 1.0
        alpha_plot = 0.95 if is_highlight else 0.55
        ax.plot(x, y, linestyle=linestyle, color=color, linewidth=linewidth, alpha=alpha_plot,
                zorder=3 if is_highlight else 1)

        # annotate at the last nonzero smoothed point
        nz = np.where(np.abs(y) > 0)[0]
        if nz.size:
            last_idx = nz[-1]
        else:
            last_idx = len(x)-1
        x_val = float(x[last_idx])
        y_val = float(y[last_idx])

        # offset in data coords
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        dx = (annotate_offset[0] / fig.dpi) * ((x1 - x0) / fig.get_size_inches()[0])
        dy = (annotate_offset[1] / fig.dpi) * ((y1 - y0) / fig.get_size_inches()[1])
        ha = 'left'
        ann_color = color
        ax.text(x_val + dx, y_val + dy, aa, color=ann_color, fontsize=9, fontweight='bold', ha=ha, va='center', zorder=4)

    ax.axhline(0, color='black', linewidth=1)
    ax.set_xlim(left=df_clean['rank'].min(), right=max_rank)
    ax.set_xlabel('Feature rank')
    ax.set_ylabel('Cumulative count (positive above, negative below)')
    ax.set_title(f'Cumulative AA counts by rank (annotated), ranks ≤ {max_rank}\n(Top by slope in first {n_init} ranks)')
    ax.grid(axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()

    if return_fig:
        return fig, ax
    return ax



In [ ]:
plot_cumulative_AA_by_sign_top_initial_slope(df_phylo_res)
